### Import Required Dependencies

In [ ]:
#Project Setup

# --- Core Libraries ---
import pandas as pd
import numpy as np
import warnings
import logging


# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Path and File Handling ---
from pathlib import Path

# --- Data Wrangling and Utilities ---
from functools import reduce
from sklearn.preprocessing import MinMaxScaler

# --- Visualization ---
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# --- Warning ---
warnings.filterwarnings('ignore')


# --- Define Project Directory (relative path for loading and saving) ---
data_dir = Path("data")     # Directory for input data
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

# --- Configure Logging ---
log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)

logging.basicConfig(
    filename=log_dir / "project.log",
    level=logging.INFO,  # Change to DEBUG for more detail
    format="%(asctime)s [%(levelname)s] %(message)s",
    filemode='w'  # overwrite log on each run
)

console = logging.StreamHandler()
console.setLevel(logging.INFO)
formatter = logging.Formatter("[%(levelname)s] %(message)s")
console.setFormatter(formatter)
logging.getLogger().addHandler(console)


print("✅ Environment ready. Data and output paths set.")


## Section 2A: Load, Clean, Pivot, and Merge National Datasets

In [ ]:
# Define directory for national datasets
complete_dir = data_dir / "complete_sets"

# Define filenames
complete_files = {
    "edu": "Education2023.csv",
    "pop": "PopulationEstimates.csv",
    "poverty": "Poverty2023.csv",
    "unemp": "Unemployment2023.csv"
}

# Fix column names BEFORE using them
df.columns = df.columns.str.strip().str.lower()
df.rename(columns={'fips code': 'fips', 'area name': 'county'}, inplace=True)

# Only now: clean county + attribute
df['county'] = df['county'].str.strip().str.lower()
df['attribute'] = df['attribute'].str.strip().str.lower()

# Load and clean each dataset
complete_data = {}

for key, filename in complete_files.items():
    path = complete_dir / filename
    try:
        df = pd.read_csv(path, encoding='cp1252')

        # Standardize column names
        df.columns = df.columns.str.strip().str.lower()

        # Rename to standard names
        df.rename(columns={
            'fips code': 'fips',
            'area name': 'county'
        }, inplace=True)

        # Clean string data
        df['county'] = df['county'].str.strip().str.lower()
        df['attribute'] = df['attribute'].str.strip().str.lower()

        # Store cleaned DataFrame
        complete_data[key] = df
        logging.info(f"✅ Loaded {filename}: {df.shape[0]} rows")

    except Exception as e:
        logging.error(f"❌ Failed to load {filename}: {e}")

# Pivot each dataset to wide format (one row per county)
try:
    edu_wide = complete_data['edu'].pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
    pop_wide = complete_data['pop'].pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
    poverty_wide = complete_data['poverty'].pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
    unemp_wide = complete_data['unemp'].pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
    logging.info("✅ Pivoted all datasets to wide format.")
except Exception as e:
    logging.error(f"❌ Pivoting failed: {e}")

# Merge the datasets on 'county'
try:
    df_full = reduce(lambda left, right: pd.merge(left, right, on='county', how='outer'),
                     [edu_wide, pop_wide, poverty_wide, unemp_wide])
    logging.info(f"✅ Final merged dataset shape: {df_full.shape}")
except Exception as e:
    logging.error(f"❌ Merge failed: {e}")


[INFO] ✅ Loaded Education2023.csv: 169245 rows
[INFO] ✅ Loaded Education2023.csv: 169245 rows
[ERROR] ❌ Failed to load PopulationEstimates.csv: 'county'
[ERROR] ❌ Failed to load PopulationEstimates.csv: 'county'
[ERROR] ❌ Failed to load Poverty2023.csv: 'county'
[ERROR] ❌ Failed to load Poverty2023.csv: 'county'
[ERROR] ❌ Failed to load Unemployment2023.csv: 'county'
[ERROR] ❌ Failed to load Unemployment2023.csv: 'county'
[ERROR] ❌ Pivoting failed: 'pop'
[ERROR] ❌ Pivoting failed: 'pop'
[INFO] ✅ Final merged dataset shape: (2102, 319)
[INFO] ✅ Final merged dataset shape: (2102, 319)


In [ ]:
# Define data directory and file paths
data_dir = Path("data")
files = [Path(f) for f in ['ed_lvl_ar.csv', 'pop_lvl_ar.csv', 'poverty_lvl_ar.csv', 'unemploy_lvl_ar.csv']]

# Load CSV files into a dictionary of DataFrames
dataframes = {file.stem: pd.read_csv(data_dir / file) for file in files}
# Load the uploaded CSV files
edu = pd.read_csv("data/ed_lvl_ar.csv")
pop = pd.read_csv("data/pop_lvl_ar.csv")
poverty = pd.read_csv("data/poverty_lvl_ar.csv")
unemp = pd.read_csv("data/unemploy_lvl_ar.csv")

# Display the first few rows and column names of each to assess structure
edu_info = edu.head(), edu.columns.tolist()
pop_info = pop.head(), pop.columns.tolist()
poverty_info = poverty.head(), poverty.columns.tolist()
unemp_info = unemp.head(), unemp.columns.tolist()

edu_info, pop_info, poverty_info, unemp_info



### Step 2B Clean and Pivot Data

In [ ]:
# Standardize column names
for df in [edu, pop, poverty, unemp]:
    df.columns = df.columns.str.strip().str.lower()

# Rename 'attributes' for consistency
edu.rename(columns={'attributes': 'attribute'}, inplace=True)

# Pivot long-format data to wide format
edu_wide = edu.pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
pop_wide = pop.pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
poverty_wide = poverty.pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
unemp_wide = unemp.pivot_table(index='county', columns='attribute', values='value', aggfunc='first')

# Merge all wide tables into one DataFrame
df_merged = reduce(lambda left, right: pd.merge(left, right, on='county', how='outer'),
                   [edu_wide, pop_wide, poverty_wide, unemp_wide])
# Display the first few rows of the merged DataFrame
df_merged.head()
df_merged.to_csv("data/merged_county_dataset.csv", index=True)


### Step 3: Inspect Dataset

In [ ]:
# Print the DataFrame information, summary statistics, and first few rows
print(df.info())
print(df.describe())
print(df.head())


#### Step 3.1 Filter to North Central Arkansas(NCA) Counties

In [ ]:
nca_counties = ['baxter', 'fulton', 'izard', 'sharp', 'marion', 'searcy']
df_nca = df_merged[df_merged.index.str.lower().isin(nca_counties)]

# Define columns of interest for analysis
education_cols = [col for col in df_nca.columns if 'bachelor' in col.lower() or 'college' in col.lower()]
poverty_cols = [col for col in df_nca.columns if 'poverty' in col.lower()]
unemployment_cols = [col for col in df_nca.columns if 'unemployment_rate' in col.lower()]
population_cols = [col for col in df_nca.columns if 'population' in col.lower() or 'census' in col.lower()]



In [ ]:
# Create standardized, renamed columns
df_nca['PovertyRate'] = df_nca['Poverty_rate_2023_y']
df_nca['UnemploymentRate'] = df_nca['Unemployment_rate_2023_y']
df_nca['BachelorsDegreeRate'] = df_nca['Bachelor\'s degree or higher, 2019-23']
df_nca['HighSchoolGradRate'] = df_nca['High school graduate or higher, 2019-23']
df_nca['Population'] = df_nca['CENSUS_2020_POP']

#### 3.2 Regional Threads

In [ ]:
# Example: Bachelor's Degree Trends
df_nca[education_cols].T.plot(kind='bar', figsize=(12, 6), title='Education Levels Across NCA Counties')
plt.ylabel("Population Estimate or %")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


### Step 4: Visualize Distributions (Histograms & KDE)

In [ ]:
# Visualize the distributions of key variables
variables = ['PovertyRate', 'UnemploymentRate', 'HighSchoolGradRate', 'BachelorsDegreeRate']

for var in variables:
    plt.figure(figsize=(8,4))
    sns.histplot(df[var], kde=True, bins=20)
    plt.title(f'Distribution of {var}')
    plt.xlabel(var)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

### Step 5: Correlation Matrix and Heatmap

In [ ]:
# Select only numeric columns of interest
corr_vars = df[['PovertyRate', 'UnemploymentRate', 'HighSchoolGradRate', 'BachelorsDegreeRate', 'Population']]
corr_matrix = corr_vars.corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Key Indicators')
plt.tight_layout()
plt.show()

### Step 6: Scatter Plots for Key Relationships

In [ ]:
# Scatter plots to visualize relationships between key indicators
plt.figure(figsize=(8,6))
sns.scatterplot(x='BachelorsDegreeRate', y='PovertyRate', data=df)
plt.title('Bachelor’s Degree Rate vs Poverty Rate')
plt.xlabel('Bachelor’s Degree (%)')
plt.ylabel('Poverty Rate (%)')
plt.tight_layout()
plt.show()

# Scatter plot for High School Grad Rate vs Unemployment Rate
plt.figure(figsize=(8,6))
sns.scatterplot(x='HighSchoolGradRate', y='UnemploymentRate', data=df)
plt.title('High School Grad Rate vs Unemployment Rate')
plt.xlabel('High School Grad (%)')
plt.ylabel('Unemployment Rate (%)')
plt.tight_layout()
plt.show()


### Step 7: Identify Outlier Counties with Boxplots

In [ ]:
# Boxplots to visualize the distribution of key indicators
for var in variables:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[var])
    plt.title(f'Boxplot of {var}')
    plt.tight_layout()
    plt.show()


### Step 8: Log-Transform Population (Optional)
If the population is skewed:

In [ ]:
df['LogPopulation'] = np.log1p(df['Population'])  # log(1 + x) avoids log(0)

plt.figure(figsize=(8,4))
sns.histplot(df['LogPopulation'], kde=True)
plt.title('Log-Transformed Population Distribution')
plt.xlabel('Log(Population)')
plt.tight_layout()
plt.show()